In [ ]:
#需要使用L4GPU，然后官方代码的专家数，从64改为16
#把依赖文件kernel.py上传
#注意本地没有GPU没法运行这个代码，要用colab或者autodl，torch版本太低不可以，因为没有rms归一化，还有本地没安装triton不可以

In [1]:
!pip list|grep triton
# 要求结果是triton 3.1.0

triton                             3.2.0


In [ ]:
!ls

kernel.py  __pycache__	sample_data


In [ ]:
import math
from dataclasses import dataclass
# 导入类型注解工具
# Tuple: 表示固定长度且各元素类型已知的元组，如 Tuple[int, float] 代表 (int, float) 这样的结构
# Optional: 表示类型可为指定类型或 None，如 Optional[int] 表示 int 或 None
# Literal: 限定变量值只能取若干字面值之一，如 Literal["a", "b"] 代表只能取 "a" 或 "b"
from typing import Tuple, Optional, Literal

import torch
from torch import nn
import torch.nn.functional as F
import torch.distributed as dist

print(torch.__version__)

from kernel import act_quant, weight_dequant, fp8_gemm

# 全局变量定义
world_size = 1  # 分布式计算中进程总数（单卡时为 1）
rank = 0        # 当前进程编号（单卡时为 0）
block_size = 128  # 可能用于矩阵运算分块优化
gemm_impl: Literal["bf16", "fp8"] = "bf16"  # GEMM 算法实现类型（bf16 或 fp8）
attn_impl: Literal["naive", "absorb"] = "absorb"  # 注意力机制实现方式

# @dataclass 装饰器可自动生成__init__、__repr__、__eq__等方法，便于管理参数与类型提示
@dataclass
class ModelArgs:
    """
    模型参数与超参数定义。

    字段说明：
        max_batch_size (int): 最大批量大小
        max_seq_len (int): 最大序列长度
        dtype (Literal["bf16", "fp8"]): 计算类型
        vocab_size (int): 词表大小
        dim (int): 隐层维度
        inter_dim (int): MLP 中间层维度
        moe_inter_dim (int): MoE 中间层维度
        n_layers (int): Transformer 层数
        n_dense_layers (int): 全连接层数
        n_heads (int): 注意力头数量
        n_routed_experts (int): MoE 路由专家数量
        n_shared_experts (int): MoE 始终激活的共享专家数量
        n_activated_experts (int): MoE 每次实际激活专家数量
        n_expert_groups (int): MoE 专家组数量
        n_limited_groups (int): MoE 路由受限组数量
        score_func (Literal["softmax", "sigmoid"]): MoE 路由打分方式
        route_scale (float): MoE 路由分数缩放因子
        q_lora_rank (int): Query 投影 LoRA 秩
        kv_lora_rank (int): Key/Value 投影 LoRA 秩
        qk_nope_head_dim (int): 无 RoPE 的 Q/K 维度
        qk_rope_head_dim (int): 含 RoPE 的 Q/K 维度
        v_head_dim (int): Value 头维度
        original_seq_len (int): RoPE 基础序列长度
        rope_theta (float): RoPE 编码基数
        rope_factor (float): RoPE 序列扩展缩放
        beta_fast (int): RoPE fast beta 超参数
        beta_slow (int): RoPE slow beta 超参数
        mscale (float): 注意力特定缩放因子
    """
    max_batch_size: int = 8  # 最大批量大小
    max_seq_len: int = 4096 * 4  # 最大序列长度
    dtype: Literal["bf16", "fp8"] = "bf16"  # 默认 bfloat16
    vocab_size: int = 102400  # 词表大小
    dim: int = 2048  # 隐藏层维度
    inter_dim: int = 10944  # MLP 中间层维
    moe_inter_dim: int = 1408  # MoE 专家中间层维
    n_layers: int = 27  # Transformer 层数
    n_dense_layers: int = 1  # 全连接层数
    n_heads: int = 16  # 注意力头数

    # MoE（Mixture of Experts）相关参数
    n_routed_experts: int = 16  # 可参与路由的 MoE 专家数量；原始论文为 256，官方默认 64，此处设 16
    n_shared_experts: int = 2   # 始终激活的共享专家数，提升模型下限能力
    n_activated_experts: int = 6  # 实际每 step 路由激活专家数
    n_expert_groups: int = 1  # 专家分组数量
    n_limited_groups: int = 1  # 路由受限组数
    score_func: Literal["softmax", "sigmoid"] = "softmax"  # MoE 路由打分方式
    route_scale: float = 1.0  # 路由分数缩放

    # Q/K/V embedding 说明：
    #   Q 总维度 = n_heads * (qk_nope_head_dim + qk_rope_head_dim)
    #   K 同上
    #   V 总维度 = n_heads * v_head_dim
    # 通常 Transformer 会设 V 头维与 Q/K 一致，此处允许 V 头维不同，可提升表示能力

    # shape 匹配说明：
    #   - 残差连接时要求主路径和残差支路张量 shape 一致；
    #   - Q/K/V 的内部 head_dim 不必完全一致，只要 Attention/MLP 输出经过投影后 shape 回到 dim 即可残差加
    #   - Residual sum, 只要所有输出最后是 [B, S, dim] 就不会 shape 错误

    # Attention 维度说明：
    #   - Q/K head_dim 必须相等，方便点积
    #   - V 可拥有独立 head_dim_v，不用等于 Q/K，可以得到不同的特征输出
    #   - 一般 Attention(V) 输出后 reshape，再做线性 projection 回到 dim，确保与输入 x 维度匹配
    #   - 常用实践见下：
    #       attn_out = Attention(Q, K, V)           # [B, S, n_heads, v_head_dim]
    #       attn_out = attn_out.reshape(B, S, -1)   # [B, S, n_heads*v_head_dim]
    #       attn_proj = attn_out @ W_o + b          # [B, S, dim]
    #       x_res = x + attn_proj                   # [B, S, dim]

    # MLA LoRA 相关参数
    q_lora_rank: int = 0      # Query LoRA 秩，0 表示关闭 LoRA，>0 开启
    kv_lora_rank: int = 512   # K/V LoRA 秩
    qk_nope_head_dim: int = 128   # 无 RoPE Q/K 头维
    qk_rope_head_dim: int = 64    # 含 RoPE Q/K 头维
    v_head_dim: int = 128         # V 头维

    # YARN（Yet Another RoPE Network）相关参数
    original_seq_len: int = 4096  # RoPE 原始序列长度
    rope_theta: float = 10000.0   # RoPE 编码基数
    rope_factor: float = 40       # RoPE 扩展缩放
    beta_fast: int = 32           # RoPE fast beta
    beta_slow: int = 1            # RoPE slow beta
    mscale: float = 1.0           # 扩展注意力缩放

2.6.0+cu124


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from typing import Optional

# dist 是 PyTorch 分布式训练模块（torch.distributed）的简称，用于支持多设备（如多台服务器或多张 GPU）之间的数据和模型并行计算。
# 通过 dist，可以实现在不同进程/设备之间同步梯度、参数、通信（比如 all_reduce、broadcast 等操作）。
# 在代码中，dist.get_world_size() 获取参与训练的进程总数（world_size），dist.get_rank() 获取当前进程的编号（rank），
# 从而实现词嵌入、模型参数等的分布式切分与同步，提升训练效率并支持更大模型。


# 假设 world_size 和 rank 由分布式训练环境提供
world_size = dist.get_world_size() if dist.is_initialized() else 1
rank = dist.get_rank() if dist.is_initialized() else 0

class ParallelEmbedding(nn.Module):
    """
    并行嵌入层（ParallelEmbedding），用于分布式训练环境下的词向量分片存储与查询。

    参数:
        vocab_size (int): 总词表大小。
        dim (int): 词向量维度。

    说明:
        - 词表会在 world_size 个进程（GPU）之间均匀分片，每个进程只持有 vocab_size // world_size 个词向量。
        - vocab_size 必须能整除 world_size。
    """
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.vocab_size = vocab_size  # 总词表大小
        self.dim = dim  # 词向量维度
        assert vocab_size % world_size == 0, f"词表大小必须能被 world_size 整除 (world_size={world_size})"

        # 每个进程负责的本地词表大小
        self.part_vocab_size = vocab_size // world_size
        # 当前进程本地词表的起止索引（在全局词表中的位置范围）
        self.vocab_start_idx = rank * self.part_vocab_size
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size
        # 当前分片的词向量参数
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播，输入为 token 索引张量 x，输出对应的词向量。

        参数:
            x (torch.Tensor): token 索引，shape 支持多维（如 [batch, seq_len]）。

        返回:
            torch.Tensor: 对应 token 的词向量，shape 为 (*x.shape, embedding_dim)。

        处理流程:
            1. 如果是多进程分布式（world_size > 1），仅查当前分片范围内的词向量，
               超出本地词表片段的索引将被设为 0（防止访问越界）。
            2. 计算本地嵌入。
            3. 若为多卡，则 all_reduce 合并输出（各进程负责的有效值会累加，形成完整结果）。
        """
        # 示例：vocab_size=16, world_size=4, rank=2, 则本卡管辖索引 8~11
        # 输入 x = tensor([7, 8, 9, 15])
        # mask = (x < 8) | (x >= 12) -> [True, False, False, True]
        # x - vocab_start_idx -> [-1, 0, 1, 7]，将不属于本片区的索引设为0
        if world_size > 1:
            # 标记输入中不属于本进程负责范围的 token
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)
            # 映射到本分片的局部索引
            x = x - self.vocab_start_idx
            # 超出范围的索引置零，避免本分片 F.embedding 越界访问
            x[mask] = 0

        # 查询本分片的 embedding
        y = F.embedding(x, self.weight)

        # 多卡情况：汇总所有分片的输出，每个 token 的真实 embedding 只由唯一分片提供，其余分片该 token 结果为零
        if world_size > 1:
            y[mask] = 0  # 保持本分片未分管部分嵌入为零，便于all_reduce
            dist.all_reduce(y)

        # 输出 y.shape = (*x.shape, embedding_dim)
        # 输入 x 若形如 (batch, seq_len)，则 y 是 (batch, seq_len, embedding_dim)
        return y


def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    线性变换函数，实现 y = xA^T + b，支持量化权重的计算。

    参数:
        x (torch.Tensor): 输入张量。
        weight (torch.Tensor): 权重张量，可能为量化格式，需要解量化处理。
        bias (Optional[torch.Tensor]): 偏置项（可选），默认为 None。

    返回:
        torch.Tensor: 线性变换后的结果。

    说明:
        - 若 weight 不是量化（element_size() > 1），直接用 F.linear 计算。
        - 若 weight 是量化的（element_size() == 1），需按 gemm_impl 不同处理：
            - gemm_impl == "bf16" 时，解量化权重后常规 F.linear。
            - 其它情况（如 fp8），先量化激活 x，再用低位 GEMM 计算。
    """
    # element_size()：PyTorch 张量的每个元素占字节数（如 float32 为4，uint8为1）。
    # 用于判断权重是否是量化格式（1字节代表如 int8/fp8 量化，非量化如 float16/32 占用2/4字节）。
    #
    # gemm_impl 控制矩阵乘法实现方式（如 "bf16"、"fp8"），依赖不同数值格式与算子优化。
    # 其值在其它配置处设置，"bf16" 表示用 bfloat16 实现，"fp8" 则用 fp8 算法。

    if weight.element_size() > 1:
        # 非量化权重，直接用常规 F.linear
        return F.linear(x, weight, bias)
    # scale 用于量化/反量化数值还原。
    #
    # 情况1（bf16分支）：weight.scale 表示量化权重对应的解量化缩放因子（一般每块一组 scale）。
    # 先经 weight_dequant 解量化恢复浮点，再用常规 F.linear 乘法。
    elif gemm_impl == "bf16":
        weight = weight_dequant(weight, weight.scale)
        return F.linear(x, weight, bias)
    # 情况2（如 fp8）：激活 x 需先量化为低位表示（act_quant 也返回缩放 scale），
    # fp8_gemm 负责低精度 GEMM，结果需加偏置还原，最终返回。
    else:
        x, scale = act_quant(x, block_size)
        y = fp8_gemm(x, scale, weight, weight.scale)
        if bias is not None:
            y += bias
        return y


class Linear(nn.Module):
    """
    自定义线性层，支持量化权重，并提供可选的偏置项。

    参数:
        in_features (int): 输入特征维度。
        out_features (int): 输出特征维度。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (可选): 计算数据类型，默认为 torch.bfloat16。

    说明:
        - 如果 weight 是量化的，则需要额外存储 scale 参数。
        - bias 可选，若不使用，则注册为 None。
    """
    dtype = torch.bfloat16  # 默认数据类型

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        super().__init__()
        self.in_features = in_features  # 输入特征维度
        self.out_features = out_features  # 输出特征维度
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))

        # 若权重是量化的（element_size() == 1），则需要 scale 参数
        if self.weight.element_size() == 1:
            scale_out_features = (out_features + block_size - 1) // block_size
            scale_in_features = (in_features + block_size - 1) // block_size
            # 存储量化 scale 参数
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))
        else:
            # 非量化情况，无需 scale 参数
            self.register_parameter("scale", None)

        # 处理偏置
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        线性层的前向传播。

        参数:
            x (torch.Tensor): 输入张量。

        返回:
            torch.Tensor: 经过线性变换的张量。

        说明:
            - 调用 linear 函数，自动处理量化权重和偏置项的计算。
        """
        return linear(x, self.weight, self.bias)


In [ ]:
"""
column并行（Column Parallel）是通过将线性层的输出特征（out_features）均匀拆分到多个GPU（分布式进程）上实现多卡并行的。在Pytorch等深度学习框架中，具体流程如下：

1. 假设有N张卡（world_size=N）和总输出特征数out_features。
2. 每张卡只负责 out_features // N 个输出特征的参数和计算。这意味着每张卡的weight参数维度变为 [out_features // N, in_features]。
3. 正向传播时，输入x在每张卡上分别与本地的weight做矩阵乘法，输出本地的输出部分 [batch, out_features // N]。
4. 如果后续需要完整的输出（例如下游全连接/softmax），可以通过all-gather等通信操作将每卡输出拼起来，得到[batch, out_features]。

这种做法的优点是，将参数和计算都分散在N张卡之间，大大减小了单卡的显存压力并加速了推理/训练。
"""
# 实际上，ColumnParallelLinear本身只是“把权重和输出分割”成多份，每个进程负责一部分权重和计算（比如 out_features // world_size）。但“在多卡（多进程、多机器）上运行”并不是 ColumnParallelLinear 这个类本身显式做的，而是依赖于 PyTorch 分布式框架进行的。
#
# 具体来说：
# 1. 分布式环境下，一般会通过 torch.distributed.launch 或 torchrun 启动多个独立的“分布式进程”，每个 GPU/进程都跑同样的代码（虽然参数不同）。
# 2. 这些进程通过 rank（当前进程编号）、world_size（总进程数），来决定“自己”负责哪块权重（例如 rank=2 就负责第3段）。
# 3. 一般来说，上下文里会有 world_size 和 rank 变量，每个进程只初始化自己负责的参数，例如:
#      my_chunk = all_chunks[rank]
#      my_linear = ColumnParallelLinear(...)
# 4. 通信（比如 all_gather 拼结果）由更高层的并行/通信库（如 deepspeed、megatron、torch.distributed）管理。
#
# 总结：真正“让权重分配到不同 GPU/进程上并运算”的关键，是你用 torchrun/torch.distributed 启动多个进程，每个进程拿到自己的 ColumnParallelLinear（只保有部分参数），并通过 rank/world_size 控制。ColumnParallelLinear 只是把这个切分逻辑实现好。

class ColumnParallelLinear(Linear):
    """
    列并行线性层（Column Parallel Linear），将输出特征分割到多个分布式进程中。

    参数：
        in_features (int): 输入特征的数量。
        out_features (int): 总输出特征数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保总输出特征数量可以被世界大小整除，以实现均匀分割
        assert out_features % world_size == 0, f"输出特征数必须能被 world_size 整除 (world_size={world_size})"

        # 计算当前进程负责的部分输出特征数
        self.part_out_features = out_features // world_size

        # 调用父类 Linear 的初始化，创建一个 in_features 到 part_out_features 的线性层
        super().__init__(in_features, self.part_out_features, bias, dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        列并行线性层的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过线性变换后的张量，进行列并行计算。
        """
        # 进行线性变换
        y = linear(x, self.weight, self.bias)
        return y


class RowParallelLinear(Linear):
    """
    行并行线性层（Row Parallel Linear），将输入特征分割到多个分布式进程中。

    参数：
        in_features (int): 总输入特征数量。
        out_features (int): 输出特征的数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保输入特征数量可以被世界大小整除，以实现均匀分割
        assert in_features % world_size == 0, f"输入特征数必须能被 world_size 整除 (world_size={world_size})"

        # 计算当前进程负责的部分输入特征数
        self.part_in_features = in_features // world_size

        # 调用父类 Linear 的初始化，创建一个 part_in_features 到 out_features 的线性层
        super().__init__(self.part_in_features, out_features, bias, dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RowParallelLinear 层的前向传播。

        参数：
            x (torch.Tensor): 输入张量，本地分段输入，形状应为 [batch_size, part_in_features]。

        返回：
            torch.Tensor: 经线性变换并全局累加得到的输出张量，形状为 [batch_size, out_features]。
        """
        # 本进程只用本地块权重做线性变换，x shape: [batch, part_in_features], weight shape: [part_in_features, out_features]
        y = linear(x, self.weight)

        # Row Parallel: 每个进程只算自己的输入分段，输出是“全输出形状”，但只有自己负责块为有效值，其余为0；
        # 通过 all_reduce(sum) 将所有进程的局部输出累加，获得全局正确输出（而不是拼接/收集）。
        # 注意：all_reduce(sum) 是按位相加，不是拼接（不是 all_gather）。
        if world_size > 1:
            dist.all_reduce(y)

        # 若有偏置，则加在 all_reduce 之后，此时每进程的 y 都是全局输出。
        if self.bias is not None:
            y += self.bias

        return y


class RMSNorm(nn.Module):
    """
    均方根归一化（RMSNorm），用于对输入张量进行归一化。

    该方法不同于标准 LayerNorm，不依赖均值，而是基于均方根（RMS）进行归一化。

    参数：
        dim (int): 输入张量的维度。
        eps (float): 用于数值稳定性的 epsilon 值，默认为 1e-6。
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim = dim  # 记录输入维度
        self.eps = eps  # 记录 epsilon 值

        # 归一化的可训练缩放参数，初始化为全 1
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor):
        """
        均方根归一化的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 归一化后的张量，保持输入形状不变。
        """
        # 使用 torch 的 rms_norm 进行均方根归一化
        # 这里的参数是缩放参数 γ（gamma），不是偏置 β。
        # RMSNorm 通常只学习一个缩放参数 weight（即 γ），
        
        # 而不像 LayerNorm 那样包含可选的偏置 β。
        # 对的，RMSNorm 的输出形式是 γ * x_norm，其中 x_norm 是归一化后的 x，不含 β 偏置项。
        # 与 LayerNorm 不同，RMSNorm 通常只含缩放参数 weight（γ），不加偏置 β。
        return F.rms_norm(x, (self.dim,), self.weight, self.eps)


In [ ]:
import torch
import torch.nn as nn
import math
from typing import Optional

# 预计算旋转位置编码的频率值
# 该函数用于计算基于旋转位置编码的复指数值
# 主要目的是为了加速计算，避免在每次前向传播时重新计算这些值
def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """
    预计算旋转位置编码的频率值。

    参数：
        args (ModelArgs): 包含位置编码参数的模型参数。

    返回：
        torch.Tensor: 预计算的复指数值，用于旋转位置编码。
    """
    dim = args.qk_rope_head_dim  # 旋转位置编码的维度,是64
    seqlen = args.max_seq_len  # 最大序列长度
    beta_fast = args.beta_fast  # 快速调整参数
    beta_slow = args.beta_slow  # 缓慢调整参数
    base = args.rope_theta  # 旋转位置编码的基数
    factor = args.rope_factor  # 旋转位置编码的缩放因子

    # 计算旋转位置编码修正维度
    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """
        计算旋转位置编码的修正维度。
        """
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))

    # 计算旋转位置编码修正范围
    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """
        计算旋转位置编码修正范围。
        """
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))
        return max(low, 0), min(high, dim-1)

    # 计算线性斜坡因子，用于平滑过渡
    def linear_ramp_factor(min, max, dim):
        """
        计算线性斜坡因子。
        """
        if min == max:
            max += 0.001
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)
        ramp_func = torch.clamp(linear_func, 0, 1)
        return ramp_func

    # 计算基础频率
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))

    # 如果序列长度超过原始最大长度，则进行修正
    if seqlen > args.original_seq_len:
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)
        freqs = freqs / factor * (1 - smooth) + freqs * smooth

    t = torch.arange(seqlen)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

# 应用旋转位置编码到输入张量
# 该函数使用预计算的复指数值对输入进行旋转编码

def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    应用旋转位置编码到输入张量。

    参数：
        x (torch.Tensor): 输入张量。
        freqs_cis (torch.Tensor): 预计算的复指数值。

    返回：
        torch.Tensor: 旋转编码后的张量。
    """
    dtype = x.dtype
    x = torch.view_as_complex(x.float().view(*x.shape[:-1], -1, 2))  # 变换为复数表示
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))  # 调整形状
    y = torch.view_as_real(x * freqs_cis).flatten(3)  # 进行旋转编码并转换回实数
    return y.to(dtype)

# 多头注意力层（MLA）  多头隐性注意力  多头潜在注意力
# 该类实现了标准的多头注意力机制，并结合了旋转位置编码
class MLA(nn.Module):
    """
    多头注意力层（MLA）。

    属性:
        dim (int): 输入特征的维度。
        n_heads (int): 注意力头的数量。
        n_local_heads (int): 分布式系统中用于局部注意力的头数量。
        q_lora_rank (int): 查询低秩投影的秩。
        kv_lora_rank (int): 键值低秩投影的秩。
        qk_nope_head_dim (int): 非位置查询/键投影的维度。
        qk_rope_head_dim (int): 旋转位置查询/键投影的维度。
        qk_head_dim (int): 查询/键投影的总维度。
        v_head_dim (int): 值投影的维度。
        softmax_scale (float): 注意力计算中Softmax的缩放因子。
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        # 初始化各个参数
        self.dim = args.dim  # 输入的特征维度
        self.n_heads = args.n_heads  # 注意力头的数量
        self.n_local_heads = args.n_heads // world_size  # 分布式环境中的局部注意力头数
        self.q_lora_rank = args.q_lora_rank  # 查询低秩投影的秩
        self.kv_lora_rank = args.kv_lora_rank  # 键值低秩投影的秩
        self.qk_nope_head_dim = args.qk_nope_head_dim  # 非位置查询/键的维度
        self.qk_rope_head_dim = args.qk_rope_head_dim  # 旋转位置查询/键的维度
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim  # 查询/键投影的总维度
        self.v_head_dim = args.v_head_dim  # 值投影的维度

        # 如果q_lora_rank为0，直接使用列并行线性层，否则使用低秩投影和标准化
        if self.q_lora_rank == 0:
            self.wq = ColumnParallelLinear(self.dim, self.n_heads * self.qk_head_dim)
        else:
            self.wq_a = Linear(self.dim, self.q_lora_rank)  # 低秩投影的第一部分，不为0，为512，先从2048变为512
            self.q_norm = RMSNorm(self.q_lora_rank)  # 低秩投影的标准化
            self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)  # 低秩投影的第二部分

        # 键值投影和标准化
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)  # 键值低秩投影
        self.kv_norm = RMSNorm(self.kv_lora_rank)  # 键值的标准化
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))  # 键值投影
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)  # 输出投影
        self.softmax_scale = self.qk_head_dim ** -0.5  # Softmax缩放因子
        
        # 如果最大序列长度大于原始序列长度，调整softmax_scale
        if args.max_seq_len > args.original_seq_len:
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0
            self.softmax_scale = self.softmax_scale * mscale * mscale

        # 根据注意力实现类型选择不同的缓存方式，这个缓存 为了加速推理
        if attn_impl == "naive":
            # 在"naive"实现下缓存k和v
            self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.qk_head_dim), persistent=False)
            self.register_buffer("v_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.v_head_dim), persistent=False)
        else:
            # 在其他实现下缓存kv和pe
            self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)
            self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        多头注意力层的前向传播。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, seq_len, dim)。
            start_pos (int): 缓存的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转位置编码（RoPE）复数指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于遮挡部分注意力计算。

        返回:
            torch.Tensor: 输出张量，形状与输入相同。
        """
        bsz, seqlen, _ = x.size()  # 取得批次大小和序列长度
        end_pos = start_pos + seqlen  # 计算本次输入在缓存中的结束位置

        # 计算查询向量 q
        if self.q_lora_rank == 0:
            q = self.wq(x)  # 列并行线性层
        else:
            q = self.wq_b(self.q_norm(self.wq_a(x)))  # 低秩 lora+归一化再线性

        # q 形状为 (bsz, seqlen, n_local_heads * qk_head_dim)
        # view 成 (bsz, seqlen, n_local_heads, qk_head_dim) 方便后续各头独立处理
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)
        print(f"q shape: {q.shape}")

        # 按照维度分割为 非PE部分 和 需要RoPE的位置部分
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)
        print(f"q_nope shape: {q_nope.shape}, q_pe shape: {q_pe.shape}")
        # 对位置编码部分应用旋转位置编码
        q_pe = apply_rotary_emb(q_pe, freqs_cis)

        # 计算键值向量 kv 和需要RoPE的 k_pe
        kv = self.wkv_a(x)
        print("kv shape:", kv.shape)
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        print(f"kv shape: {kv.shape}, k_pe shape: {k_pe.shape}")

        # k_pe 的 unsqueeze(2): 形状从 [bsz, seqlen, qk_rope_head_dim] 变为 [bsz, seqlen, 1, qk_rope_head_dim]
        # 这样方便与 freqs_cis (如[1, seqlen, 1, qk_rope_head_dim]) 按位置广播，用于 apply_rotary_emb
        # 此处第三维 1 不是头数，仅为广播相容，RoPE 默认每个 head 共享一套编码（如需每 head 独立编码需手动 repeat/expand）
        print(f"freqs_cis shape: {freqs_cis.shape}")
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)

        # 注意力实现类型分支：naive 使用 head-wise kv 缓存, absorb用低秩 kv 缓存
        if attn_impl == "naive":
            # 拼接非位置和位置编码部分的查询向量
            q = torch.cat([q_nope, q_pe], dim=-1)
            # 低秩投影后的 kv -> 归一化 -> 列并行线性层
            kv = self.wkv_b(self.kv_norm(kv))
            # 转为 (bsz, seqlen, n_local_heads, qk_nope_head_dim + v_head_dim)
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)
            # Split 为键 k_nope 和值 v
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)
            # 拼接 k 的非位置与位置编码部分，k_pe 扩展头数
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)
            print(f"k shape: {k.shape}")
            # 更新缓存
            self.k_cache[:bsz, start_pos:end_pos] = k
            self.v_cache[:bsz, start_pos:end_pos] = v
            # 计算注意力分数
            scores = torch.einsum("bshd,bthd->bsht", q, self.k_cache[:bsz, :end_pos]) * self.softmax_scale
        else:
            # 处理 wkv_b 权重，按 n_local_heads 切片（因分布式张量并行）
            wkv_b = self.wkv_b.weight if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size)
            # view 成 (n_local_heads, out_dim, kv_lora_rank)
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)
            # 只对 no position encoding 的头做线性映射
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            # 更新低秩 kv 和 RoPE 编码缓存
            self.kv_cache[:bsz, start_pos:end_pos] = self.kv_norm(kv)
            self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)
            # 注意力分数 = no position的kv内积 + 位置编码内积
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale

        # 如果有mask，添加到得分上
        if mask is not None:
            scores += mask.unsqueeze(1)

        # 计算softmax得分
        scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)

        # 根据注意力实现类型，选择不同的计算方式
        if attn_impl == "naive":
            x = torch.einsum("bsht,bthd->bshd", scores, self.v_cache[:bsz, :end_pos])  # 计算输出
        else:
            # 这里分两步计算是因为需要先使用注意力权重对低秩缓存 kv_cache 做加权和（第一步），
            # 得到有效的 value 表示（shape: bsz, seqlen, n_local_heads, kv_lora_rank），
            # 然后再通过 wkv_b 的 value 映射（选取最后 v_head_dim 部分），做一次线性变换（第二步），
            # 得到最终输出。两步拆开便于分别理解注意力加权和输出映射。

            # 这一步其实是实现了类似传统注意力里的 "对V加权求和"，但这里的V不是最终形式，
            # 而是对低秩"键值缓存" (kv_cache) 先加权——也就是query和low-rank键值的attention分数乘积，
            # 这一步相当于：对于每个query，将kv_cache（低秩特征）根据注意力分数聚合，
            # 结果不是最终的输出，而是后续再用 wkv_b 的 value 投影做一次线性组合才变成真正的V。
            # 这样设计是为了节省显存和提升效率（低秩分解），所以不直接保存大体积的V而是低秩分量，最后动态生成。
            # 换言之，这里的attention先让query和"紧凑低秩表示"做点积，再用特定映射生成最终value。

            # 第一步：用注意力权重（scores）对 kv_cache 做加权，聚合信息至每个 token
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])  # 计算加权和
            
            # 第二步：将加权结果通过 value 映射（wkv_b 的最后 v_head_dim 通道），实现线性投影得到最终输出
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])  # 计算最终输出

        # 通过行并行线性层投影到原始维度
        x = self.wo(x.flatten(2))
        return x



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple

class MLP(nn.Module):
    """
    多层感知机（MLP），用于前馈计算。

    该模块包含三个线性变换层，分别是 w1、w2 和 w3，用于特征变换和计算。

    属性:
        w1 (nn.Module): 线性层，用于从输入层到隐藏层的转换。
        w2 (nn.Module): 线性层，用于从隐藏层到输出层的转换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 MLP 层。

        参数:
            dim (int): 输入和输出的维度（维度保持一致）。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()
        self.w1 = ColumnParallelLinear(dim, inter_dim)  # 第一层线性变换
        self.w2 = RowParallelLinear(inter_dim, dim)     # 第二层线性变换（回到原始维度）
        self.w3 = ColumnParallelLinear(dim, inter_dim)  # 额外的线性变换层
        
    # 这里没有 seq 这一维，是因为 MLP 层只对每个位置（token）的特征维度进行独立计算，不涉及序列维度的交互。
    # 在前传时，nn.Linear (如 self.w1, self.w2, self.w3) 默认作用在最后一维（特征维），
    # 无论输入有无 batch 或 seq 维，都会自动广播和作用在每个 token 上。
    # 所以 forward 方法中的输入 x 可以是 (batch_size, seq_len, dim) 或 (batch_size, dim)，
    # 实际线性层只转换最后一维 dim，seq 维不会在初始化参数里单独体现。

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MLP 的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            torch.Tensor: 经过 MLP 计算后的输出张量，形状为 (batch_size, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # 使用 SiLU 激活函数进行非线性变换，并结合 w3(x) 进行特征变换


class Gate(nn.Module):
    """
    用于 Mixture-of-Experts（MoE）模型的门控机制（Gating Mechanism）。

    该模块用于在多个专家（Expert）之间进行路由选择，决定每个输入数据应该被送到哪些专家进行计算。

    属性:
        dim (int): 输入特征的维度。
        topk (int): 每个输入激活的专家数（选择 top-k 个专家）。
        n_groups (int): 专家被划分的组数（用于路由分组）。
        topk_groups (int): 每个输入路由到的专家组数。
        score_func (str): 计算分数的函数（可选 "softmax" 或 "sigmoid"）。
        route_scale (float): 路由权重的缩放因子。
        weight (torch.nn.Parameter): 可训练参数，表示门控网络的权重矩阵。
        bias (Optional[torch.nn.Parameter]): 可选的偏置项，仅当输入维度为 7168 时存在。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Gate 模块。

        参数:
            args (ModelArgs): 传入的模型参数对象，包含 MoE 相关的超参数。
        """
        super().__init__()
        self.dim = args.dim  # 输入特征的维度
        self.topk = args.n_activated_experts  # 选择的前 top-k 个专家
        self.n_groups = args.n_expert_groups  # 专家分组的数量
        self.topk_groups = args.n_limited_groups  # 选择的前 top-k 组
        self.score_func = args.score_func  # 计算分数的方式
        self.route_scale = args.route_scale  # 路由权重的缩放比例

        # 可训练权重参数（用于计算门控分数）
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))

        # 只有当 dim = 7168 时，才会添加可训练的偏置项
        self.bias = nn.Parameter(torch.empty(args.n_routed_experts)) if self.dim == 7168 else None

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        计算门控权重，并确定选择哪些专家进行计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            Tuple[torch.Tensor, torch.Tensor]:
                - 选择的专家权重 (batch_size, topk)
                - 选择的专家索引 (batch_size, topk)
        """
        # 一行代码f格式打印x的形状
        print(f"Gate x shape: {x.shape}")
        scores = linear(x, self.weight)  # 计算输入与门控权重的线性变换

        # 根据 score_func 选择 softmax 或 sigmoid 进行归一化
        if self.score_func == "softmax":
            scores = scores.softmax(dim=-1, dtype=torch.float32)
        else:
            scores = scores.sigmoid()

        original_scores = scores  # 保存原始分数

        # scores 是二维 (batch_size, n_routed_experts)，bias 是一维 (n_routed_experts)，可以直接广播相加
        
        if self.bias is not None:
            scores = scores + self.bias

        # 举例说明这个分组过程是怎么操作的
        # 假设：batch_size=2，n_groups=3，每组有4个专家（即 n_routed_experts=12）
        # x 的 shape: (2, dim)
        # scores 的 shape: (2, 12)
        # 
        # 首先将 scores reshape 为 (2, 3, 4)，即 (batch_size, n_groups, 每组专家数)
        # 
        # 举个例子，
        # 假设 scores = 
        # tensor([
        #   [0.1, 0.2, 0.1, 0.3,  0.5, 0.6, 0.1, 0.2,  0.4, 0.3, 0.2, 0.1],
        #   [0.8, 0.7, 0.6, 0.5,  0.4, 0.3, 0.2, 0.1,  0.9, 0.8, 0.7, 0.6]
        # ])
        # reshape 后变为 (2, 3, 4):
        # tensor([
        #   [[0.1, 0.2, 0.1, 0.3], [0.5, 0.6, 0.1, 0.2], [0.4, 0.3, 0.2, 0.1]],
        #   [[0.8, 0.7, 0.6, 0.5], [0.4, 0.3, 0.2, 0.1], [0.9, 0.8, 0.7, 0.6]]
        # ])
        #
        # 然后对每一组做汇总（max、top2和等），得到每个 group 的分数
        # 比如 group_scores = scores.amax(dim=-1) 得到 (2, 3)
        # 
        # 接着选 topk_groups（比如2）个得分最高的 group，mask 其它 group（置零），
        # 最后把 shape 拉回到 (2, 12)，只保留被选中组的专家分数。

        # 继续上面的例子，比较 amax 和 top2 的效果
        # 假设 group_scores = scores.amax(dim=-1)
        # 得到的是每组中分值最大的专家的分数，例如：
        # 第一组 [0.1, 0.2, 0.1, 0.3] -> 0.3
        # 第二组 [0.5, 0.6, 0.1, 0.2] -> 0.6
        # 第三组 [0.4, 0.3, 0.2, 0.1] -> 0.4
        # 最终 group_scores = [0.3, 0.6, 0.4]

        # 如果用 group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)
        # 则取每组中分值最大的两个专家并求和，例如：
        # 第一组 top2: [0.2, 0.3] -> 0.2 + 0.3 = 0.5
        # 第二组 top2: [0.6, 0.5] -> 0.6 + 0.5 = 1.1
        # 第三组 top2: [0.4, 0.3] -> 0.4 + 0.3 = 0.7
        # 最终 group_scores = [0.5, 1.1, 0.7]
        # 也就是说，amax 只关心本组中最大的专家分数，top2 则关注本组中得分前两个专家的总和，可能更鲁棒

        # 专家分组的目的主要是为了限制每个 token 最多只激活 topk_groups 个 group，从而间接控制最多多少专家能被选中
        # 或者说分组后，每组内可有多个专家参与，具备更均匀或多样的路由效果，避免所有 token 都集中到同一小部分专家
        # 分组处理后，后续的 topk 操作是对 *被激活的所有专家* 里再次选 topk
        # 也就是说，分组掩码已经在 scores 上把未选 group 的分数归零（或者梯度阻断），
        # 而最终 topk 是在 flatten 后的 (batch_size, n_experts) 里挑 topk 个分数最高的专家
        # 所以两步不会冲突：分组筛掉大部分 group，topk 再细挑本批 token 真正激活的 topk 专家
        # 如果没有分组，topk 就是直接所有专家里选，不做分片限制

        # 若使用多个专家组，则进行分组处理
        if self.n_groups > 1:
            scores = scores.view(x.size(0), self.n_groups, -1)  # 重新 reshape 为 (batch_size, n_groups, 每组的专家数)

            # 计算每组的得分，若无偏置，则取最大值；若有偏置，则取 top-2 得分之和
            if self.bias is None:
                group_scores = scores.amax(dim=-1)
            else:
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)

            # 选择得分最高的 topk_groups 组，并生成掩码
            # 这里 group_scores 的形状为 (batch_size, n_groups)
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]

            # # 这里需要根据 topk_groups 的分组结果，mask 只保留被选中组的得分，屏蔽其他组。
            # # scores 形状: [batch_size, n_groups, n_experts_per_group]
            # # indices: 每个 batch 选中 topk_groups 组的 group 下标, 形状 [batch_size, topk_groups]
            # # 
            # # 1. 构造 group 级别的 mask [batch_size, n_groups]，选中的位置为 True，其余为 False
            # group_mask = torch.zeros_like(scores[..., 0], dtype=torch.bool) # [batch_size, n_groups]
            # group_mask.scatter_(1, indices, True)                          # 只在 topk_groups 位置为 True

            # # 2. 扩展 mask 形状到 [batch_size, n_groups, 1]，与 scores 对齐，便于做逐组逐专家的掩码
            # expert_mask = group_mask.unsqueeze(-1)                         # [batch_size, n_groups, 1]

            # # 3. mask 掉未被选中的 group，选中的组保留原 score，未选中的组所有专家的分数为 0
            # masked_scores = scores * expert_mask                           # [batch_size, n_groups, n_experts_per_group]

            # # 4. 恢复回原始分布 (flatten 到 [batch_size, n_groups*n_experts_per_group])
            # scores = masked_scores.flatten(1)                              # [batch_size, n_experts]
            # 下面是对每一步语法用法的解释

            # 1. torch.zeros_like(scores[..., 0]) 创建一个和 scores[..., 0] 相同形状的全零张量。
            #    scores[..., 0] 其实就是把 scores 的最后一维去掉（只看 group 维度），形状为 [batch_size, n_groups]
            # 2. .scatter_(1, indices, True) 是在第1维（即 n_groups 这一维）按照 indices 指定的位置，把值置为 True（布尔类型的 True 即 1）。
            #    这样就把 topk_groups 组的那个位置设为 True（被选中），其他全是 False。
            mask = torch.zeros_like(scores[..., 0]).scatter_(1, indices, True)

            # 3. mask.unsqueeze(-1) 把 mask 末尾加一个维度，变为 [batch_size, n_groups, 1]
            #    这样（通过广播）乘 scores 时，每组的所有专家：被选中组位置是1，其他组位置是0。
            # 4. 把未被选中组的所有专家分数都变成0（mask 掉了）。
            # 5. .flatten(1) 是把 n_groups 与每组的专家数展平成一维，使 scores 最后回到 (batch_size, n_experts)
            scores = (scores * mask.unsqueeze(-1)).flatten(1)  # 仅保留选中的专家分数

        # 选择得分最高的 top-k 个专家
        indices = torch.topk(scores, self.topk, dim=-1)[1]
        #打印indices
        print("Gate indices shape:", indices.shape)
        # 计算最终的专家权重
        weights = original_scores.gather(1, indices)
        #打印weights
        # print("Gate weights :", weights)
        # 若使用 sigmoid，则需要归一化
        if self.score_func == "sigmoid":
            weights /= weights.sum(dim=-1, keepdim=True)

        weights *= self.route_scale  # 乘以路由缩放因子
        return weights.type_as(x), indices  # 返回计算后的权重和选择的专家索引


class Expert(nn.Module):
    """
    专家（Expert）层，用于 Mixture-of-Experts（MoE）模型。

    该模块实现了一个独立的专家网络，每个专家由三层线性变换层组成。

    属性:
        w1 (nn.Module): 线性层，从输入到隐藏层的变换。
        w2 (nn.Module): 线性层，从隐藏层到输出的变换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 Expert 层。

        参数:
            dim (int): 输入和输出的维度。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()
        self.w1 = nn.Linear(dim, inter_dim)  # 输入到隐藏层的线性变换
        self.w2 = nn.Linear(inter_dim, dim)  # 隐藏层到输出层的线性变换
        self.w3 = nn.Linear(dim, inter_dim)  # 额外的线性变换层

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Expert 层的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            torch.Tensor: 经过 Expert 计算后的输出张量，形状为 (batch_size, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # 先经过 w1 进行非线性变换，再与 w3 计算的结果相乘，最后通过 w2 输出


In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from typing import Optional

class MoE(nn.Module):
    """
    MoE（Mixture-of-Experts，专家混合）模块，用于选择性地激活多个专家网络，提高计算效率。

    主要属性：
        dim (int): 输入特征的维度。
        n_routed_experts (int): 该模型中的总专家数量。
        n_local_experts (int): 在分布式环境中，每个设备处理的专家数量。
        n_activated_experts (int): 每个输入激活的专家数量。
        gate (nn.Module): 用于计算输入到各专家的分配权重的门控机制。
        experts (nn.ModuleList): 专家网络列表，每个专家都是一个神经网络模块。
        shared_experts (nn.Module): 共享专家网络，对所有输入均生效。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 MoE 模块。

        参数：
            args (ModelArgs): 包含 MoE 参数的模型配置。
        """
        super().__init__()
        self.dim = args.dim

        # 确保专家数量可以被世界大小整除（用于分布式训练）。
        assert args.n_routed_experts % world_size == 0, f"专家数量必须被 world_size 整除 (world_size={world_size})"

        self.n_routed_experts = args.n_routed_experts
        self.n_local_experts = args.n_routed_experts // world_size
        self.n_activated_experts = args.n_activated_experts

        # 计算当前设备负责的专家索引范围。
        self.experts_start_idx = rank * self.n_local_experts
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts

        # 门控机制，用于决定输入数据分配给哪些专家。
        self.gate = Gate(args)

        # 仅在当前设备上初始化其负责的专家，其余设为 None。
        self.experts = nn.ModuleList([Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx else None
                                      for i in range(self.n_routed_experts)])

        # 共享专家，对所有输入均适用。
        self.shared_experts = MLP(args.dim, args.n_shared_experts * args.moe_inter_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MoE 模块的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过专家计算后的输出张量。
        """
        
        shape = x.size()
        # 打印输入 x 的形状
        print(f"MoE x shape: {x.shape}")
        # 将输入展平成二维 [token数, dim]，方便专家选择
        x = x.view(-1, self.dim)

        # 通过门控获取每个 token 被选中的专家索引及其权重
        weights, indices = self.gate(x)
        y = torch.zeros_like(x)

        # 统计每个专家被分配的 token 数目
        # counts[i] 表示第 i 号专家在本 batch 被分配到的 token 数量
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()

        # 遍历本地（当前设备负责的）专家，为分配到 token 的专家做前向与加权累加
        for i in range(self.experts_start_idx, self.experts_end_idx):
            # 若该专家未被分配输入则跳过
            if counts[i] == 0:
                continue

            expert = self.experts[i]

            # 查找所有被当前专家 i 选中的 token 下标及其在 topk 门控中的列号
            idx, top = torch.where(indices == i)
            print(f'moe-idx{idx}')

            # 1. x[idx]：取出属于专家 i 的全部 token 输入
            # 2. expert(x[idx])：专家输出，形状[N, dim]
            # 3. weights[idx, top, None]：对这些 token 的权重，shape [N, 1]
            # 4. 广播乘权并累加写回 y[idx]
            y[idx] += expert(x[idx]) * weights[idx, top, None]

        # 所有输入统一经过共享专家（无门控）
        z = self.shared_experts(x)

        # 多卡时需要全局同步专家输出 y
        if world_size > 1:
            dist.all_reduce(y)
        # 打印最终专家（y）与共享专家（z）的输出形状
        print(f"MoE y shape: {y.shape}, MoE z shape: {z.shape}")
        # 还原回原始 (batch, seqlen, dim) 形状，保证 token 顺序正确
        # flatten->专家分配->write-back流程均保持顺序一致，只变展平形状
        #
        # 结果是：最终输出 = 各 token 的专家输出（加权求和）+ 共享专家输出
        # 这样兼具动态专家表达和统一全局表达
        return (y + z).view(shape)




import torch
import torch.nn as nn
import torch.distributed as dist
from typing import Optional

class Block(nn.Module):
    """
    Transformer 块，结合了注意力机制和前馈网络。

    属性:
        attn (nn.Module): 多头注意力（MLA, Multi-Head Attention）。
        ffn (nn.Module): 前馈神经网络（MLP 或 MoE）。
        attn_norm (nn.Module): 注意力层的归一化层。
        ffn_norm (nn.Module): 前馈网络的归一化层。
    """
    def __init__(self, layer_id: int, args: ModelArgs):
        """
        初始化 Transformer 块。

        参数:
            layer_id (int): 该层在 Transformer 中的索引。
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        super().__init__()
        self.attn = MLA(args)  # 多头潜在注意力机制
        # 如果 layer_id 小于稠密层数量，则使用 MLP，否则使用 MoE 结构
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)
        self.attn_norm = RMSNorm(args.dim)  # 注意力归一化层
        self.ffn_norm = RMSNorm(args.dim)   # 前馈归一化层

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Transformer 块的前向传播。

        参数:
            x (torch.Tensor): 输入张量。
            start_pos (int): 序列的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于排除特定位置的注意力。

        返回:
            torch.Tensor: 经过 Transformer 块计算后的输出张量。
        """
        x = x + self.attn(self.attn_norm(x), start_pos, freqs_cis, mask)  # 归一化后进行注意力计算并残差连接
        x = x + self.ffn(self.ffn_norm(x))  # 归一化后进入前馈网络并残差连接
        return x


class Transformer(nn.Module):
    """
    Transformer 模型，包括嵌入层、多层 Transformer 块、最终归一化层和输出层。

    属性:
        max_seq_len (int): 最大序列长度。
        embed (nn.Module): 词嵌入层。
        layers (torch.nn.ModuleList): Transformer 块的列表。
        norm (nn.Module): 所有 Transformer 层之后的归一化层。
        head (nn.Module): 输出投影层，将隐藏状态映射到词汇表大小。
        freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Transformer 模型。

        参数:
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        global world_size, rank
        world_size = dist.get_world_size() if dist.is_initialized() else 1  # 获取分布式训练的总进程数
        rank = dist.get_rank() if dist.is_initialized() else 0  # 获取当前进程的 rank 值
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16  # 设置默认数据类型
        super().__init__()
        self.max_seq_len = args.max_seq_len  # 最大序列长度
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)  # 词嵌入层，支持并行计算
        self.layers = torch.nn.ModuleList()
        for layer_id in range(args.n_layers):
            self.layers.append(Block(layer_id, args))  # 添加多个 Transformer 块
        self.norm = RMSNorm(args.dim)  # 归一化层
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.get_default_dtype())  # 输出投影层
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)  # 预计算旋转位置编码

    @torch.inference_mode()
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        """
        Transformer 的前向传播。

        参数:
            tokens (torch.Tensor): 形状为 (batch_size, seq_len) 的输入 token ID。
            start_pos (int, 可选): 序列的起始位置，默认为 0。

        返回:
            torch.Tensor: 形状为 (batch_size, vocab_size) 的 logits。
        """
        seqlen = tokens.size(1)  # 获取输入序列长度
        h = self.embed(tokens)  # 通过词嵌入层获取 token 表示
        freqs_cis = self.freqs_cis[start_pos:start_pos+seqlen]  # 获取对应位置的旋转位置编码
        mask = None
        if seqlen > 1:
            mask = torch.full((seqlen, seqlen), float("-inf"), device=tokens.device).triu_(1)  # 构造上三角掩码（防止未来信息泄露）
        for layer in self.layers:
            h = layer(h, start_pos, freqs_cis, mask)  # 依次通过每个 Transformer 块
        h = self.norm(h)[:, -1]  # 归一化后取最后一个时间步的输出
        logits = self.head(h)  # 通过输出投影层计算 logits
        if world_size > 1:
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]
            dist.all_gather(all_logits, logits)  # 在所有进程间收集 logits
            logits = torch.cat(all_logits, dim=-1)  # 拼接所有进程的 logits
        return logits


if __name__ == "__main__":
    torch.set_default_dtype(torch.bfloat16)  # 设置默认数据类型
    torch.set_default_device("cuda")  # 设置默认计算设备为 GPU
    torch.manual_seed(0)  # 设置随机种子，保证可复现性
    args = ModelArgs()  # 初始化模型参数
    x = torch.randint(0, args.vocab_size, (2, 128))  # 随机生成 token ID 作为输入
    model = Transformer(args)  # 初始化 Transformer 模型
    print(model(x).size())  # 运行模型并打印输出张量的形状，为啥只有一个词，因为只取了最后一个的输出，可以理解这个样例像咱们的bert基座一样，让大家去练习一个分类问题



q shape: torch.Size([2, 128, 16, 192])
q_nope shape: torch.Size([2, 128, 16, 128]), q_pe shape: torch.Size([2, 128, 16, 64])
kv shape: torch.Size([2, 128, 576])
kv shape: torch.Size([2, 128, 512]), k_pe shape: torch.Size([2, 128, 64])
q shape: torch.Size([2, 128, 16, 192])
q_nope shape: torch.Size([2, 128, 16, 128]), q_pe shape: torch.Size([2, 128, 16, 64])
kv shape: torch.Size([2, 128, 576])
kv shape: torch.Size([2, 128, 512]), k_pe shape: torch.Size([2, 128, 64])
MoE x shape: torch.Size([2, 128, 2048])
Gate x shape: torch.Size([256, 2048])
Gate indices shape: torch.Size([256, 6])
Gate weights : tensor([[0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        ...,
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625]], device='cuda:0',
       dtype=torc